# Build `synthetic_berlin` pairwise comparisons from `SurveyResults_200414.json`

This notebook:
1. Loads the Straßencheck export JSON (`SurveyResults_200414.json`) as specified in *Spezifikation – Ausgabeformat des Straßenchecks*. fileciteturn0file0  
2. Converts within-session Likert ratings (0–3) into **paired comparisons** (left/right/tie) following the standard transformation to paired comparisons described by Dittrich et al. fileciteturn0file1  
3. Writes:
   - `synthetic_berlin_comparisons_df.pickle` (only synthetic)
   - `comparisons_df_with_synthetic_berlin.pickle` (existing + synthetic)
4. Prints dataset statistics.
5. Optionally downsamples to a target size (e.g., 7,500 comparisons).
6. Downloads all referenced images to `/home/csantiago/images/synthetic_berlin`.

**Run location**: this notebook assumes it is executed from: `/home/csantiago/build_datasets`.


In [1]:
# --- Imports ---
import os
import json
import math
import random
from pathlib import Path
from typing import Dict, Any, List, Tuple, Iterable, Optional

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("OK: imports loaded")

OK: imports loaded


/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 0) Paths and constants

The export JSON is described as a list of **sessions**; each session contains `session_id`, `created`, `profile`, and a list of `ratings`, each with `scene_id`, `duration`, and `rating` (0=unsicher … 3=sicher). fileciteturn0file0

We treat the Likert rating as an **ordinal safety score** where higher means “safer”.  
Paired comparison label convention used here matches your project:

- `score = -1`  → **left** image is safer  
- `score = 0`   → tie  
- `score = +1`  → **right** image is safer


In [2]:
# --- User-configurable paths ---
# Notebook runs from: /home/csantiago/build_datasets

JSON_PATH = Path("SurveyResults_200414.json")  # adjust if needed
EXISTING_COMPARISONS_PATH = Path("../comparisons_df.pickle")  # adjust if needed

# Outputs
OUT_SYNTHETIC_PATH = Path("synthetic_berlin_comparisons_df.pickle")
OUT_MERGED_PATH = Path("comparisons_df_with_synthetic_berlin.pickle")

# Image download config
IMAGE_ROOT = Path("/home/csantiago/images/synthetic_berlin")
BASE_URL = "https://fmb-aws-bucket.s3.eu-central-1.amazonaws.com/KatasterKI/"  # base from your example

assert JSON_PATH.exists(), f"Missing JSON file: {JSON_PATH.resolve()}"
print("JSON_PATH:", JSON_PATH.resolve())
print("EXISTING_COMPARISONS_PATH exists:", EXISTING_COMPARISONS_PATH.exists())
print("Outputs:", OUT_SYNTHETIC_PATH.resolve(), OUT_MERGED_PATH.resolve())

JSON_PATH: /home/csantiago/build_datasets/SurveyResults_200414.json
EXISTING_COMPARISONS_PATH exists: True
Outputs: /home/csantiago/build_datasets/synthetic_berlin_comparisons_df.pickle /home/csantiago/build_datasets/comparisons_df_with_synthetic_berlin.pickle


## 1) Load and inspect the Straßencheck export JSON

Each JSON entry is one session (a single user run), with `ratings` being the list of scene ratings. fileciteturn0file0


In [3]:
def load_sessions(json_path: Path) -> List[Dict[str, Any]]:
    with json_path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Expected top-level JSON to be a list of sessions.")
    return data

sessions = load_sessions(JSON_PATH)
print("Number of sessions:", len(sessions))
print("First session keys:", list(sessions[0].keys()))
print("Example ratings entry keys:", list((sessions[0].get("ratings") or [{}])[0].keys()))

Number of sessions: 21481
First session keys: ['session_id', 'project', 'profile', 'created', 'stopped_at_scene_id', 'ratings']
Example ratings entry keys: ['scene_id', 'duration', 'rating']


## 2) Helper functions: SceneID → image URL/path

The Spezifikation defines `scene_id` as the SceneID “as coded in the survey specification” and notes that it also encodes perspective/experiment. fileciteturn0file0

In practice (based on your example), we expect the *important part* to look like:
`scenes/01_MS_C_573.jpg`.

This notebook handles common cases:
- `scene_id` already contains `.jpg` → use it (and ensure it is under `scenes/` if needed).
- `scene_id` has no extension → append `.jpg` and prefix with `scenes/`.

If your export uses a different encoding, adjust `scene_id_to_relpath`.


In [4]:
def scene_id_to_relpath(scene_id: str) -> str:
    """Return relative path like 'scenes/<...>.jpg' suitable to append to BASE_URL."""
    if not isinstance(scene_id, str) or len(scene_id.strip()) == 0:
        return ""
    s = scene_id.strip()

    # If already a path containing 'scenes/' keep it, otherwise enforce prefix.
    if s.lower().endswith(".jpg") or s.lower().endswith(".jpeg"):
        if "scenes/" not in s:
            s = f"scenes/{s}"
        return s

    # If no extension, assume jpg.
    if "scenes/" in s:
        return f"{s}.jpg"
    return f"scenes/{s}.jpg"

def relpath_to_url(relpath: str) -> str:
    if not relpath:
        return ""
    return BASE_URL.rstrip("/") + "/" + relpath.lstrip("/")

# Quick sanity check using your example
example = "01_MS_C_573.jpg"
print(scene_id_to_relpath(example))
print(relpath_to_url(scene_id_to_relpath(example)))

scenes/01_MS_C_573.jpg
https://fmb-aws-bucket.s3.eu-central-1.amazonaws.com/KatasterKI/scenes/01_MS_C_573.jpg


## 3) Transform Likert ratings → paired comparisons

Dittrich et al. describe the standard transformation: for each respondent/session, for each pair of items (i, j), define a comparison outcome based on whether the Likert response is higher, equal, or lower. fileciteturn0file1

Here, each “item” is a *scene* (an image). Within a session:
- collect all rated scenes (drop missing)
- for every pair of scenes, produce a comparison label

Notes:
- We keep metadata: `dataset`, `survey_id`, `trial_id`, `session_created`, `perspective` (if inferable), `rating_l`, `rating_r`.
- `survey_id`: we use `session_id` (the session identifier in the export). fileciteturn0file0
- `trial_id`: unique identifier per generated comparison within a session (`<session_id>_<k>`).


In [5]:
def infer_perspective_from_scene_id(scene_id: str) -> Optional[str]:
    # Many IDs look like 01_MS_C_573 where C is perspective. We'll try to parse.
    if not isinstance(scene_id, str):
        return None
    base = Path(scene_id).stem  # strip extension if present
    parts = base.split("_")
    # heuristic: last or third token might be perspective letter
    for token in parts:
        if token in {"C", "A", "P"}:
            return token
    return None

def make_pair_label(r_left: int, r_right: int) -> int:
    # Higher rating = safer
    if r_left > r_right:
        return -1
    elif r_left < r_right:
        return 1
    return 0

def session_to_comparisons(session: Dict[str, Any]) -> List[Dict[str, Any]]:
    sid = session.get("session_id")
    created = session.get("created")
    ratings = session.get("ratings") or []
    rows = []

    # Extract rated scenes
    items = []
    for r in ratings:
        scene_id = r.get("scene_id")
        rating_val = r.get("rating")
        if scene_id is None or rating_val is None:
            continue
        try:
            rating_val = int(rating_val)
        except Exception:
            continue
        relpath = scene_id_to_relpath(str(scene_id))
        url = relpath_to_url(relpath)
        items.append({
            "scene_id": str(scene_id),
            "relpath": relpath,
            "url": url,
            "rating": rating_val,
            "duration_ms": int(r.get("duration", 0) or 0),
            "perspective": infer_perspective_from_scene_id(str(scene_id)),
        })

    # Remove duplicates inside a session (keep first occurrence)
    seen = set()
    uniq = []
    for it in items:
        key = it["relpath"]
        if key in seen:
            continue
        seen.add(key)
        uniq.append(it)

    n = len(uniq)
    if n < 2:
        return rows

    # Generate all within-session pairs
    k = 0
    for i in range(n):
        for j in range(i + 1, n):
            left = uniq[i]
            right = uniq[j]
            score = make_pair_label(left["rating"], right["rating"])

            rows.append({
                "dataset": "synthetic_berlin",
                "image_l": Path(left["relpath"]).name,   # store filename to match your existing conventions
                "image_r": Path(right["relpath"]).name,
                "url_l": left["url"],
                "url_r": right["url"],
                "score": int(score),
                "rating_l": int(left["rating"]),
                "rating_r": int(right["rating"]),
                "survey_id": str(sid) if sid is not None else None,
                "trial_id": f"{sid}_{k}" if sid is not None else f"none_{k}",
                "session_created": created,
                "perspective_l": left["perspective"],
                "perspective_r": right["perspective"],
            })
            k += 1
    return rows

# Quick test on 1 session
test_rows = session_to_comparisons(sessions[0])
print("Pairs from first session:", len(test_rows))
pd.DataFrame(test_rows).head()

Pairs from first session: 435


,dataset,image_l,image_r,url_l,url_r,score,rating_l,rating_r,survey_id,trial_id,session_created,perspective_l,perspective_r
0,synthetic_berlin,01_MS_C_344.jpg,01_CP_C_1226.jpg,https://fmb-aws-bucket.s3.eu-central-1.amazona...,https://fmb-aws-bucket.s3.eu-central-1.amazona...,0,3,3,6bbf3fed-46ab-4221-930b-4ac3fa0acff8,6bbf3fed-46ab-4221-930b-4ac3fa0acff8_0,2019-12-01T17:31:14.489910Z,C,C
1,synthetic_berlin,01_MS_C_344.jpg,01_SE_C_54.jpg,https://fmb-aws-bucket.s3.eu-central-1.amazona...,https://fmb-aws-bucket.s3.eu-central-1.amazona...,-1,3,1,6bbf3fed-46ab-4221-930b-4ac3fa0acff8,6bbf3fed-46ab-4221-930b-4ac3fa0acff8_1,2019-12-01T17:31:14.489910Z,C,C
2,synthetic_berlin,01_MS_C_344.jpg,01_SE_C_26.jpg,https://fmb-aws-bucket.s3.eu-central-1.amazona...,https://fmb-aws-bucket.s3.eu-central-1.amazona...,-1,3,2,6bbf3fed-46ab-4221-930b-4ac3fa0acff8,6bbf3fed-46ab-4221-930b-4ac3fa0acff8_2,2019-12-01T17:31:14.489910Z,C,C
3,synthetic_berlin,01_MS_C_344.jpg,01_MS_C_290.jpg,https://fmb-aws-bucket.s3.eu-central-1.amazona...,https://fmb-aws-bucket.s3.eu-central-1.amazona...,-1,3,1,6bbf3fed-46ab-4221-930b-4ac3fa0acff8,6bbf3fed-46ab-4221-930b-4ac3fa0acff8_3,2019-12-01T17:31:14.489910Z,C,C
4,synthetic_berlin,01_MS_C_344.jpg,01_MS_C_451.jpg,https://fmb-aws-bucket.s3.eu-central-1.amazona...,https://fmb-aws-bucket.s3.eu-central-1.amazona...,0,3,3,6bbf3fed-46ab-4221-930b-4ac3fa0acff8,6bbf3fed-46ab-4221-930b-4ac3fa0acff8_4,2019-12-01T17:31:14.489910Z,C,C


## 4) Build the full `synthetic_berlin` comparisons DataFrame

This may be large (potentially millions of pairs). The code uses a streaming approach to avoid holding everything in memory at once: it builds a list of chunks and concatenates periodically.


In [6]:
def build_synthetic_df(
    sessions: List[Dict[str, Any]],
    chunk_size_sessions: int = 200,
    max_sessions: Optional[int] = None,
) -> pd.DataFrame:
    out_chunks = []
    buffer = []

    total = len(sessions) if max_sessions is None else min(len(sessions), max_sessions)
    for idx, sess in enumerate(tqdm(sessions[:total], desc="Sessions")):
        buffer.extend(session_to_comparisons(sess))

        if (idx + 1) % chunk_size_sessions == 0 and buffer:
            out_chunks.append(pd.DataFrame(buffer))
            buffer = []

    if buffer:
        out_chunks.append(pd.DataFrame(buffer))

    if not out_chunks:
        return pd.DataFrame(columns=["dataset","image_l","image_r","score","survey_id","trial_id"])

    df = pd.concat(out_chunks, ignore_index=True)
    return df

# Build. If you want to test quickly, set max_sessions=200.
synthetic_df = build_synthetic_df(sessions, chunk_size_sessions=200, max_sessions=None)

print("Synthetic comparisons:", len(synthetic_df))
synthetic_df.head()

Sessions: 100%|██████████| 21481/21481 [00:36<00:00, 586.94it/s]


Synthetic comparisons: 6500567


,dataset,image_l,image_r,url_l,url_r,score,rating_l,rating_r,survey_id,trial_id,session_created,perspective_l,perspective_r
0,synthetic_berlin,01_MS_C_344.jpg,01_CP_C_1226.jpg,https://fmb-aws-bucket.s3.eu-central-1.amazona...,https://fmb-aws-bucket.s3.eu-central-1.amazona...,0,3,3,6bbf3fed-46ab-4221-930b-4ac3fa0acff8,6bbf3fed-46ab-4221-930b-4ac3fa0acff8_0,2019-12-01T17:31:14.489910Z,C,C
1,synthetic_berlin,01_MS_C_344.jpg,01_SE_C_54.jpg,https://fmb-aws-bucket.s3.eu-central-1.amazona...,https://fmb-aws-bucket.s3.eu-central-1.amazona...,-1,3,1,6bbf3fed-46ab-4221-930b-4ac3fa0acff8,6bbf3fed-46ab-4221-930b-4ac3fa0acff8_1,2019-12-01T17:31:14.489910Z,C,C
2,synthetic_berlin,01_MS_C_344.jpg,01_SE_C_26.jpg,https://fmb-aws-bucket.s3.eu-central-1.amazona...,https://fmb-aws-bucket.s3.eu-central-1.amazona...,-1,3,2,6bbf3fed-46ab-4221-930b-4ac3fa0acff8,6bbf3fed-46ab-4221-930b-4ac3fa0acff8_2,2019-12-01T17:31:14.489910Z,C,C
3,synthetic_berlin,01_MS_C_344.jpg,01_MS_C_290.jpg,https://fmb-aws-bucket.s3.eu-central-1.amazona...,https://fmb-aws-bucket.s3.eu-central-1.amazona...,-1,3,1,6bbf3fed-46ab-4221-930b-4ac3fa0acff8,6bbf3fed-46ab-4221-930b-4ac3fa0acff8_3,2019-12-01T17:31:14.489910Z,C,C
4,synthetic_berlin,01_MS_C_344.jpg,01_MS_C_451.jpg,https://fmb-aws-bucket.s3.eu-central-1.amazona...,https://fmb-aws-bucket.s3.eu-central-1.amazona...,0,3,3,6bbf3fed-46ab-4221-930b-4ac3fa0acff8,6bbf3fed-46ab-4221-930b-4ac3fa0acff8_4,2019-12-01T17:31:14.489910Z,C,C


## 5) Statistics for `synthetic_berlin`

This cell prints key descriptive stats:
- number of unique sessions/users (by `survey_id`)
- unique images
- number of comparisons
- label distribution
- per-user comparisons summary


In [7]:
def print_synthetic_stats(df: pd.DataFrame) -> None:
    df = df.copy()
    df["pair"] = df["image_l"].astype(str) + "||" + df["image_r"].astype(str)

    n = len(df)
    n_users = df["survey_id"].nunique(dropna=True)
    unique_images = pd.unique(pd.concat([df["image_l"], df["image_r"]], ignore_index=True))
    n_images = len([x for x in unique_images if isinstance(x, str) and x != "nan"])

    print("=== synthetic_berlin stats ===")
    print("comparisons:", n)
    print("unique users/sessions (survey_id):", n_users)
    print("unique images (filenames):", n_images)

    print("\nLabel distribution (score):")
    print(df["score"].value_counts(dropna=False).sort_index())

    per_user = df.groupby("survey_id").size()
    print("\nPer-user comparisons:")
    print(per_user.describe(percentiles=[0.1,0.25,0.5,0.75,0.9]).to_string())

print_synthetic_stats(synthetic_df)

=== synthetic_berlin stats ===
comparisons: 6500567
unique users/sessions (survey_id): 21210
unique images (filenames): 3018

Label distribution (score):
score
-1    2164503
 0    2421341
 1    1914723
Name: count, dtype: int64

Per-user comparisons:
count     21210.000000
mean        306.485950
std        1264.350041
min           1.000000
10%         105.000000
25%         105.000000
50%         190.000000
75%         300.000000
90%         435.000000
max      146611.000000


## 6) Downsample to a target number of comparisons (e.g., 7,500)

This cell can reduce the dataset size for faster experimentation.
- Stratified by `score` to preserve label distribution (approximately).
- Uses `SEED` for reproducibility.
- Writes `synthetic_berlin_comparisons_df_N<target>.pickle`.

If you want a stricter constraint (e.g., max comparisons per user or per image), extend the sampler accordingly.


In [16]:
TARGET_N = 20000  # change to whatever you want
OUT_DOWNSAMPLED = Path(f"synthetic_berlin_comparisons_df_N{TARGET_N}.pickle")

def stratified_downsample(df: pd.DataFrame, target_n: int, seed: int = 42) -> pd.DataFrame:
    if target_n >= len(df):
        return df.copy()

    rng = np.random.default_rng(seed)
    parts = []
    counts = df["score"].value_counts(normalize=True)

    for label, frac in counts.items():
        n_label = int(round(frac * target_n))
        sub = df[df["score"] == label]
        if len(sub) <= n_label:
            parts.append(sub)
        else:
            idx = rng.choice(sub.index.to_numpy(), size=n_label, replace=False)
            parts.append(df.loc[idx])

    out = pd.concat(parts, ignore_index=True)

    # Fix rounding drift
    if len(out) > target_n:
        out = out.sample(n=target_n, random_state=seed).reset_index(drop=True)
    elif len(out) < target_n:
        # top up uniformly
        missing = target_n - len(out)
        remaining = df.drop(out.index, errors="ignore")
        if len(remaining) > 0:
            top = remaining.sample(n=min(missing, len(remaining)), random_state=seed)
            out = pd.concat([out, top], ignore_index=True)

    return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)

synthetic_df_small = stratified_downsample(synthetic_df, TARGET_N, seed=SEED)
print("Downsampled size:", len(synthetic_df_small))
print(synthetic_df_small["score"].value_counts().sort_index())

synthetic_df_small.to_pickle(OUT_DOWNSAMPLED)
print("Wrote:", OUT_DOWNSAMPLED.resolve())

Downsampled size: 20000
score
-1    6659
 0    7450
 1    5891
Name: count, dtype: int64
Wrote: /home/csantiago/build_datasets/synthetic_berlin_comparisons_df_N20000.pickle


## 7) Save synthetic dataset and merge with your existing `comparisons_df.pickle`

We create:
- `synthetic_berlin_comparisons_df.pickle`
- `synthetic_berlin_comparisons_df_N{dpwnsample}`
- `comparisons_df_with_synthetic_berlin.pickle`

If your existing pickle uses different column names, adjust the harmonization section.


In [18]:
# Save synthetic-only-downsampled
synthetic_df_small.to_pickle(OUT_DOWNSAMPLED)
print("Wrote:", OUT_DOWNSAMPLED.resolve())

# Save synthetic-only
#synthetic_df.to_pickle(OUT_SYNTHETIC_PATH)
#print("Wrote:", OUT_SYNTHETIC_PATH.resolve())

# Merge with existing (if present)
if EXISTING_COMPARISONS_PATH.exists():
    base_df = pd.read_pickle(EXISTING_COMPARISONS_PATH)
    print("Loaded existing comparisons:", len(base_df))

    # Harmonize minimal expected schema
    # Ensure these columns exist in both
    for col in ["dataset", "image_l", "image_r", "score"]:
        if col not in base_df.columns:
            base_df[col] = None

    merged_df = pd.concat([base_df, synthetic_df_small], ignore_index=True)
    merged_df.to_pickle(OUT_MERGED_PATH)
    print("Wrote merged:", OUT_MERGED_PATH.resolve(), "rows:", len(merged_df))
else:
    print("Existing comparisons pickle not found; skipping merge.")

Wrote: /home/csantiago/build_datasets/synthetic_berlin_comparisons_df_N20000.pickle
Loaded existing comparisons: 13623
Wrote merged: /home/csantiago/build_datasets/comparisons_df_with_synthetic_berlin.pickle rows: 33623


## 8) Download all referenced images to `/home/csantiago/images`

This downloads the **filenames** `image_l` and `image_r` using `url_l` and `url_r`.
- Skips files that already exist.
- Uses concurrent downloads for speed.
- By default, it downloads images referenced in the **downsampled** dataset (if it exists); you can switch to `synthetic_df` if you want all.

If you run this on millions of comparisons, you will still only download each unique image once.


In [10]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

#DOWNLOAD_FROM = "downsampled"  # "downsampled" or "full"
#df_for_download = synthetic_df_small if (DOWNLOAD_FROM == "downsampled") else synthetic_df
df_for_download = synthetic_df   # ALWAYS download from full dataset

IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

# Build unique (filename, url) mapping
pairs = pd.concat([
    df_for_download[["image_l", "url_l"]].rename(columns={"image_l": "fname", "url_l": "url"}),
    df_for_download[["image_r", "url_r"]].rename(columns={"image_r": "fname", "url_r": "url"}),
], ignore_index=True).dropna()

pairs = pairs.drop_duplicates(subset=["fname"]).reset_index(drop=True)

print("Unique images to consider:", len(pairs))

def download_one(fname: str, url: str, out_dir: Path, timeout: int = 30) -> Tuple[str, bool, str]:
    out_path = out_dir / fname
    if out_path.exists() and out_path.stat().st_size > 0:
        return fname, True, "exists"

    try:
        r = requests.get(url, stream=True, timeout=timeout)
        r.raise_for_status()
        tmp = out_path.with_suffix(out_path.suffix + ".tmp")
        with tmp.open("wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 256):
                if chunk:
                    f.write(chunk)
        tmp.replace(out_path)
        return fname, True, "downloaded"
    except Exception as e:
        return fname, False, str(e)

max_workers = min(32, (os.cpu_count() or 8) * 2)
results = {"downloaded": 0, "exists": 0, "failed": 0}

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futs = []
    for row in pairs.itertuples(index=False):
        futs.append(ex.submit(download_one, row.fname, row.url, IMAGE_ROOT))

    for fut in tqdm(as_completed(futs), total=len(futs), desc="Downloading"):
        fname, ok, msg = fut.result()
        if ok and msg == "downloaded":
            results["downloaded"] += 1
        elif ok and msg == "exists":
            results["exists"] += 1
        else:
            results["failed"] += 1

print("Download summary:", results)
print("Images directory:", IMAGE_ROOT)

Unique images to consider: 3018


Downloading: 100%|██████████| 3018/3018 [00:00<00:00, 24937.27it/s]

Download summary: {'downloaded': 0, 'exists': 3018, 'failed': 0}
Images directory: /home/csantiago/images/synthetic_berlin
